# Phase 7a — Hierarchical Drill-Down (Python data POV)

Per the multi-stage plan (see AGENTS.md). This notebook shows the
**schemas**, **raw data**, **BAML functions**, and **useful key
parts of the pipeline** that populate the CopilotKit-backed web route
at `/drill-down` (Phase 7b).

Three drill levels:
1. Pick a subnation -> see all stages
2. Pick a stage -> see all subjects
3. Pick a subject -> see topics + equivalencies + extracted Markdown

In [ ]:
# 1. Show the canonical Phase 7 schemas.
from cocoindex_flows.pdf._shared import extract_markdown, output_path_for
from cocoindex_flows.education.lc6_extraction_app import ExtractionRow
from cocoindex_flows.equivalency.equivalency_graph_app import (
    TopicNode, TopicEquivalentEdge, CANONICAL_JURISDICTIONS,
)
from experiments.prompt_sweeps.prompt_sweep import (
    SUBJECT_CLIENTS, ALL_8,
)
print("Phase 7 schemas:")
print(f"  ExtractionRow fields: {list(ExtractionRow.__dataclass_fields__.keys())}")
print(f"  TopicNode fields: {list(TopicNode.__dataclass_fields__.keys())}")
print(f"  TopicEquivalentEdge fields: {list(TopicEquivalentEdge.__dataclass_fields__.keys())}")
print(f"  Subjects: {ALL_8}")
print(f"  Canonical jurisdictions: {CANONICAL_JURISDICTIONS}")

In [ ]:
# 2. Show raw rows from the Phase 3 extracted_syllabi + Phase 4 equivalency tables.
import pathlib, sqlite3, json

SQLITE = pathlib.Path("data/bi_ep/extracted_syllabi.sqlite")
if SQLITE.exists():
    with sqlite3.connect(str(SQLITE)) as conn:
        rows = conn.execute(
            "SELECT subnation, subject_slug, language, "
            "substr(syllabus_json, 1, 60) "
            "FROM extracted_syllabi LIMIT 5"
        ).fetchall()
    print(f"Phase 3 extracted_syllabi rows: {len(rows)}")
    for r in rows:
        print(f"  {r[0]:20s} {r[1]:20s} {r[2]:3s}  {r[3]}...")

    with sqlite3.connect(str(SQLITE)) as conn:
        try:
            edges = conn.execute(
                "SELECT source_subnation, target_subnation, confidence "
                "FROM topic_equivalent_edges LIMIT 5"
            ).fetchall()
        except sqlite3.OperationalError:
            edges = []
    print(f"\nPhase 4 topic_equivalent_edges sample: {len(edges)}")
    for e in edges:
        print(f"  {e[0]:18s} -> {e[1]:18s}  confidence={e[2]:.2f}")
else:
    print("No SQLite database yet. Run Phases 3 + 4 first.")

In [ ]:
# 3. Show BAML functions that produce the data (signature + docstring).
try:
    from baml_client import b  # type: ignore[import-not-found]
    import inspect as _inspect

    for fn_name in (
        "ExtractCurriculumSyllabus",
        "ExtractExamPaperLayout",
        "ExtractMarkingSchemeGuideline",
        "ExtractCrossLinguisticConcept",
        "ExtractSyllabusDiagram",
    ):
        fn = getattr(b, fn_name, None)
        if fn is None:
            continue
        sig = _inspect.signature(fn)
        print(f"\n{fn_name}{sig}")
        doc = (fn.__doc__ or "").strip().split("\n")[0]
        print(f"  doc: {doc}")
except ImportError:
    print("baml_client not installed (canonical dev path).")
    print("In production: from baml_client import b")
    print("Then call b.ExtractCurriculumSyllabus(pdf_text=..., subject=..., language=...)")

In [ ]:
# 4. Useful key parts of the pipeline (what feeds the CopilotKit web route).
print("Canonical pipeline parts that populate /drill-down:")
print()
print("  dlt_pipelines/official_doc_fetcher.py")
    print("    -> 10 jurisdictional @dlt.resource blocks -> official_documents DuckDB table")
print()
print("  dlt_pipelines/pdf_downloader.py  (Phase 2a)")
    print("    -> downloads the 7 remote-URL PDFs to data/bi_ep/syllabi_raw/<src>/<subj>/<lang>/<sha>.pdf")
print()
print("  cocoindex_flows/pdf/pdf_to_markdown_app.py  (Phase 2b)")
    print("    -> walks data/bi_ep/syllabi_raw/ -> emits .md to data/bi_ep/syllabi_md/<src>/<subj>/<lang>/<sha>.md")
print()
print("  cocoindex_flows/education/lc6_extraction_app.py  (Phase 3)")
    print("    -> runs all 5 LC6 BAML functions -> extracted_syllabi table (composite PK)")
print()
print("  cocoindex_flows/equivalency/equivalency_graph_app.py  (Phase 4)")
    print("    -> topic_nodes + topic_equivalent_edges (cross-jurisdiction graph)")
print()
print("  experiments/model_comparison/runner.py  (Phase 5)")
    print("    -> EvalResult rows for the 5-model harness")
print()
print("  experiments/prompt_sweeps/prompt_sweep.py  (Phase 6)")
    print("    -> SweepResult rows measuring prompt_overlay lift per subject")
print()
print("  web/src/routes/drill-down.tsx  (Phase 7b, the user-facing surface)")
    print("    -> 3-level drill: subnation -> stage -> subject")
    print("    -> Reads extracted_syllabi + topic_equivalent_edges + the markdown files")

## Summary

- This notebook shows the **schemas** (ExtractionRow, TopicNode, TopicEquivalentEdge) + the **canonical 8 LC subjects** + the **6 canonical jurisdictions**.
- It pulls **raw rows** from the Phase 3 (extracted_syllabi) + Phase 4 (topic_equivalent_edges) tables so you can see the actual data shape.
- It enumerates the **BAML function signatures** so you know exactly which extractor feeds which column.
- It maps every **useful key part of the pipeline** to the stage that produces the data, ending with the `/drill-down` web route (Phase 7b).
- Phase 7b ships the **CopilotKit-backed web route** at `web/src/routes/drill-down.tsx` — same AG-UI SSE bridge + A2UI catalog, but the drill surface is hierarchical (subnation → stage → subject) instead of chat-style.
- Ready for Phase 8 (local dev + dev Cloud Run deploy) — the wrap-up.